# 🏋️‍♂️ CalorieVision — LSTM Classifier Training Notebook (Google Colab)

This notebook trains the PyTorch LSTM exercise classifier for CalorieVision across 26 exercise classes.

### Key Features:
1. **Landmark Normalization**: Body-centered (hip-midpoint shift) and scale-normalized (torso height) to ensure translation and scale invariance.
2. **Dataset Shuffling**: Random permutation before train/validation split so all 26 classes are present in both splits.
3. **Gradient Clipping**: Prevents exploding gradients during sequence training.

In [ ]:
# Step 1: Install Dependencies
!pip install torch numpy mediapipe matplotlib

In [ ]:
import json
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

# 26 Exercise Classes
EXERCISE_CLASSES = [
    'squat', 'pushup', 'jumping_jack', 'lunge', 'plank', 'burpee',
    'mountain_climber', 'high_knees', 'situp', 'jump_rope', 'bicycle_crunch', 'shoulder_press',
    'deadlift', 'pull_up', 'bench_press', 'tricep_dip', 'leg_raise', 'wall_sit',
    'box_jump', 'russian_twist', 'hip_thrust', 'calf_raise', 'lateral_raise', 'bicep_curl',
    'kettlebell_swing', 'superman_hold'
]
NUM_CLASSES = len(EXERCISE_CLASSES)
INPUT_DIM = 99  # 33 landmarks * (x, y, z)
SEQ_LEN = 30    # 30 frames per window (1 second at 30 fps)
STRIDE = 15     # 50% overlap between windows

print(f'Total Classes: {NUM_CLASSES}')

In [ ]:
# Step 2: Define PyTorch LSTM Classifier Architecture
class LSTMClassifier(nn.Module):
    def __init__(self, input_dim=99, hidden_dim=128, num_layers=2, num_classes=26, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        # x shape: (batch_size, seq_len, input_dim)
        out, (h_n, c_n) = self.lstm(x)
        last_out = out[:, -1, :]
        logits = self.fc(last_out)
        return logits

In [ ]:
# Step 3: Function to slice raw frames into normalized 30-frame sliding windows
def frames_to_windows(frames, seq_len=30, stride=15):
    n_frames = len(frames)
    if n_frames == 0:
        return np.empty((0, seq_len, 99), dtype=np.float32), []

    X_matrix = np.zeros((n_frames, 99), dtype=np.float32)
    for i, f in enumerate(frames):
        lms = f.get('landmarks', [])
        if lms and len(lms) >= 33:
            try:
                coords = np.array([[lm['x'], lm['y'], lm['z']] for lm in lms[:33]], dtype=np.float32)
                # 1. Hip center normalization (landmarks 23 left_hip, 24 right_hip)
                hip_center = (coords[23] + coords[24]) / 2.0
                coords -= hip_center
                # 2. Torso scale normalization (distance from shoulder center to hip center)
                shoulder_center = (coords[11] + coords[12]) / 2.0
                torso_height = float(np.linalg.norm(shoulder_center))
                if torso_height > 1e-4:
                    coords /= torso_height
                X_matrix[i] = coords.flatten()
            except (KeyError, TypeError, IndexError):
                pass

    windows = []
    start_indices = []
    for start in range(0, max(1, n_frames - seq_len + 1), stride):
        end = start + seq_len
        if end > n_frames:
            break
        windows.append(X_matrix[start:end])
        start_indices.append(start)

    return np.array(windows, dtype=np.float32), start_indices

In [ ]:
# Step 4: Load training dataset (Upload combined_keypoints.json & combined_labels.json)
from google.colab import files

print('Upload dataset/combined_keypoints.json and dataset/combined_labels.json:')
try:
    uploaded = files.upload()
except Exception:
    uploaded = {}

kp_key = next((k for k in uploaded if 'keypoint' in k.lower()), None)
lbl_key = next((k for k in uploaded if 'label' in k.lower()), None)

if kp_key and lbl_key:
    print(f'Loading dataset from {kp_key} and {lbl_key}...')
    data = json.loads(uploaded[kp_key].decode('utf-8'))
    frames = data.get('frames', data)
    per_frame_labels = json.loads(uploaded[lbl_key].decode('utf-8'))
    X_windows, start_indices = frames_to_windows(frames)
    y_labels = []
    for start in start_indices:
        end = min(start + 30, len(per_frame_labels))
        w_lbls = [EXERCISE_CLASSES.index(lbl) for lbl in per_frame_labels[start:end] if lbl in EXERCISE_CLASSES]
        y_labels.append(Counter(w_lbls).most_common(1)[0][0] if w_lbls else 0)
    X = X_windows
    y = np.array(y_labels, dtype=np.int64)
else:
    print('No uploaded dataset found. Generating synthetic demonstration dataset...')
    rng = np.random.default_rng(seed=42)
    X = rng.random((500, 30, 99)).astype(np.float32)
    y = rng.integers(0, NUM_CLASSES, size=500).astype(np.int64)

print(f'Training dataset X shape: {X.shape}, y shape: {y.shape}')

In [ ]:
# Step 5: Train LSTM Classifier & Download Checkpoint
# SHUFFLE dataset randomly before train/val split so all classes are present in train & val
perm = np.random.default_rng(seed=42).permutation(len(X))
X_shuffled, y_shuffled = X[perm], y[perm]

split_idx = int(len(X) * 0.8)
X_train, X_val = torch.from_numpy(X_shuffled[:split_idx]), torch.from_numpy(X_shuffled[split_idx:])
y_train, y_val = torch.from_numpy(y_shuffled[:split_idx]), torch.from_numpy(y_shuffled[split_idx:])

loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
model = LSTMClassifier(num_classes=NUM_CLASSES)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

best_loss = float('inf')
best_state = None

epochs = 50
for epoch in range(1, epochs + 1):
    model.train()
    for xb, yb in loader:
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_logits = model(X_val)
        val_loss = criterion(val_logits, y_val).item()
        val_acc = (val_logits.argmax(1) == y_val).float().mean().item()

    if val_loss < best_loss:
        best_loss = val_loss
        best_state = {k: v.cpu() for k, v in model.state_dict().items()}

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:2d}/{epochs} - val_loss: {val_loss:.4f} - val_acc: {val_acc:.4f}')

# Save model checkpoint
checkpoint_data = {
    'model_state_dict': best_state or model.state_dict(),
    'classes': EXERCISE_CLASSES,
    'input_dim': 99,
    'hidden_dim': 128,
    'num_layers': 2,
}
torch.save(checkpoint_data, 'lstm_best.pt')
print('✅ Saved checkpoint -> lstm_best.pt')

try:
    files.download('lstm_best.pt')
except Exception:
    print('Checkpoint saved to lstm_best.pt')